In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "retail_dev")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsentregafinal")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = (
    f"abfss://{container}@{storageName}.dfs.core.windows.net/"
    "ordenes/ordenes_raw.json"
)

print(ruta)

In [0]:
ordenes_schema = StructType(fields=[
    StructField("orden_id", StringType(), True),
    StructField("cliente_id", StringType(), True),
    StructField("fecha_orden", StringType(), True),
    StructField("fecha_entrega", StringType(), True),
    StructField("estatus", StringType(), True),
    StructField("metodo_pago", StringType(), True),
    StructField("canal", StringType(), True),
    StructField("estado_envio", StringType(), True),
    StructField("costo_envio", StringType(), True)
])

In [0]:
df_ordenes_read = (
    spark.read
    .option("multiLine", "true")
    .schema(ordenes_schema)
    .json(ruta)
)

In [0]:
display(df_ordenes_read)

In [0]:
df_ordenes_ingestion_date = (
    df_ordenes_read
    .withColumn("ingestion_date", current_timestamp())
)

In [0]:
df_ordenes_final = df_ordenes_ingestion_date.select(
    "orden_id",
    "cliente_id",
    "fecha_orden",
    "fecha_entrega",
    "estatus",
    "metodo_pago",
    "canal",
    "estado_envio",
    "costo_envio",
    "ingestion_date"
)

In [0]:
df_ordenes_final.write.mode("overwrite").insertInto(
    f"{catalogo}.{esquema}.ordenes"
)

In [0]:
display(
    spark.table(f"{catalogo}.{esquema}.ordenes")
)

In [0]:
cantidad_origen = df_ordenes_read.count()

cantidad_bronze = (
    spark.table(f"{catalogo}.{esquema}.ordenes")
    .count()
)

print(f"Registros leídos del archivo: {cantidad_origen}")
print(f"Registros cargados en Bronze: {cantidad_bronze}")

if cantidad_origen != cantidad_bronze:
    raise ValueError(
        "La cantidad de registros del origen no coincide con Bronze."
    )

print("Carga de órdenes completada correctamente.")